# Programming Assignment: Twitter Sentiment Analysis
## Building a Real-World Social Media Sentiment Classifier

---

## 📋 Assignment Information

**Course:** Unstructured Data Analysis  
**Assignment:** Programming Assignment – Twitter Sentiment Analysis  
**Deadline:** November 25, 2025, 23:59  

---

## 🎯 Objectives

In this assignment, you will:
- Implement Twitter-specific text preprocessing (emojis, hashtags, mentions, URLs)
- Build and enhance lexicon-based sentiment classifiers for social media text
- Engineer Twitter-specific features and train machine learning models
- Fine-tune BERT for short social media text
- Create a practical brand monitoring system
- Compare and analyze different sentiment analysis approaches

---

## 📝 Instructions

### Step 1: Access the Assignment

1. **Open the Google Colab notebook:**
   - Click "Open with Google Colaboratory" at the top
   - If you don't see this option, right-click → Open with → Google Colaboratory

2. **Make your own copy:**
   - Go to `File` → `Save a copy in Drive`
   - This creates your personal copy that you can edit

### Step 2: Complete the Assignment

1. **Fill in the blanks:**
   - Find all code cells marked with `# TODO:`
   - Replace each blank (`# YOUR CODE HERE`) with the correct code
   - There are **15 TODO sections** throughout the notebook

2. **Run all cells:**
   - Execute cells **in order** from top to bottom
   - Make sure there are no errors
   - Verify that your outputs make sense

3. **Answer discussion questions:**
   - Throughout the notebook, you'll find discussion questions
   - Add your answers in markdown cells
   - Provide thoughtful, analytical responses based on your results

### Step 3: Save Your Work

1. **Rename your file:**
   - Go to `File` → `Rename`
   - Use this **exact format**: `TwitterSentiment_YourName_YourStudentID.ipynb`
   - Example: `TwitterSentiment_JohnDoe_2024123456.ipynb`

2. **Download your notebook:**
   - Go to `File` → `Download` → `Download .ipynb`
   - Save it to your computer

### Step 4: Submit

1. **Upload to the course platform:**
   - Navigate to the **Assignments** section on the course website
   - Find **"Programming Assignment – Twitter Sentiment Analysis"**
   - Upload your `.ipynb` file

2. **Verify submission:**
   - Make sure the file uploaded successfully
   - Check that the filename follows the required format
   - Confirm the submission deadline: **November 25, 23:59**

---

## Setup and Installation

Run this cell to install required packages.

In [ ]:
# Install required packages
!pip install -q emoji wordcloud
!pip install -q nltk textblob
!pip install -q transformers torch scikit-learn

import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)

print("✓ Setup complete!")

## Import Libraries

In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

# Text processing
import emoji
from wordcloud import WordCloud

# NLP libraries
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

# Deep Learning
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("✓ All libraries imported successfully!")

---
# Part 1: Data Loading & Twitter Preprocessing

Twitter text is different from traditional text:
- **Short**: ~80-140 characters
- **Noisy**: typos, slang, abbreviations
- **Special features**: emojis, hashtags, @mentions, URLs

You'll build a preprocessing pipeline that handles these Twitter-specific elements.

## TODO 1: Load and Explore Dataset

Load the Twitter sentiment dataset and explore its characteristics.

In [ ]:
# TODO 1: Load the dataset
# Expected columns: 'text', 'sentiment'
# Solution: Create sample Twitter data for demonstration
sample_tweets = [
    ("I love this product! 😍 #amazing @Company", 1),
    ("This is absolutely terrible 😡", 0),
    ("Best purchase ever! Highly recommend 👍", 1),
    ("Waste of money. Very disappointed 😞", 0),
    ("Amazing quality! Will buy again ❤️ #happy", 1),
    ("Horrible customer service @Company", 0),
    ("Perfect! Exactly what I needed 🎉", 1),
    ("Not worth it. Save your money 👎", 0),
    ("Excellent product quality! #recommend", 1),
    ("Completely broken. Do not buy! 😠", 0),
] * 100  # Replicate to have reasonable dataset size

df = pd.DataFrame(sample_tweets, columns=['text', 'sentiment'])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle

# Display basic information
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
display(df.head())

# Display sentiment distribution
print(f"\nSentiment distribution:")
print(df['sentiment'].value_counts())

# Calculate statistics
df['text_length'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

print(f"\nAverage tweet length: {df['text_length'].mean():.1f} characters")
print(f"Average word count: {df['word_count'].mean():.1f} words")

# Visualize sentiment distribution
plt.figure(figsize=(8, 5))
df['sentiment'].value_counts().plot(kind='bar')
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

## TODO 2: Implement Twitter Preprocessor

Create a class that extracts Twitter-specific features and cleans the text appropriately.

In [ ]:
class TwitterPreprocessor:
    """
    Comprehensive Twitter text preprocessing.
    """

    def __init__(self, remove_urls=True, remove_mentions=True, keep_emojis=True, keep_hashtags=True):
        self.remove_urls = remove_urls
        self.remove_mentions = remove_mentions
        self.keep_emojis = keep_emojis
        self.keep_hashtags = keep_hashtags

    def extract_features(self, text):
        """
        Extract Twitter-specific features.
        """
        # TODO 2a: Extract URLs using regex pattern r'http\S+|www\S+'
        urls = re.findall(r'http\S+|www\S+', text)
        
        # TODO 2b: Extract mentions using regex pattern r'@\w+'
        mentions = re.findall(r'@\w+', text)
        
        # TODO 2c: Extract hashtags using regex pattern r'#\w+'
        hashtags = re.findall(r'#\w+', text)
        
        # Extract emojis
        emojis = [c['emoji'] for c in emoji.emoji_list(text)]

        return {
            'urls': urls,
            'mentions': mentions,
            'hashtags': hashtags,
            'emojis': emojis,
            'has_url': len(urls) > 0,
            'has_mention': len(mentions) > 0,
            'has_hashtag': len(hashtags) > 0,
            'has_emoji': len(emojis) > 0,
            'emoji_count': len(emojis)
        }

    def clean_text(self, text):
        """
        Clean and normalize tweet text.
        """
        cleaned = text

        # TODO 2d: Remove or replace URLs if self.remove_urls is True
        if self.remove_urls:
            cleaned = re.sub(r'http\S+|www\S+', '', cleaned)
        
        # TODO 2e: Remove or replace mentions if self.remove_mentions is True
        if self.remove_mentions:
            cleaned = re.sub(r'@\w+', '', cleaned)
        
        # Process hashtags (keep word, remove #)
        if self.keep_hashtags:
            cleaned = re.sub(r'#(\w+)', r'\1', cleaned)
        else:
            cleaned = re.sub(r'#\w+', '', cleaned)

        # Lowercase
        cleaned = cleaned.lower()

        # Remove extra whitespace
        cleaned = ' '.join(cleaned.split())

        return cleaned

    def preprocess(self, text):
        """
        Full preprocessing: extract features + clean text.
        """
        features = self.extract_features(text)
        cleaned_text = self.clean_text(text)

        return {
            'text': cleaned_text,
            'features': features
        }

# Test the preprocessor
preprocessor = TwitterPreprocessor()

test_tweet = "I love this! 😍 #amazing @Company https://example.com"
result = preprocessor.preprocess(test_tweet)

print("Original tweet:")
print(test_tweet)
print("\nCleaned text:")
print(result['text'])
print("\nExtracted features:")
print(result['features'])

In [ ]:
# Apply preprocessing to entire dataset
print("Preprocessing all tweets...")
results = [preprocessor.preprocess(text) for text in df['text']]

df['cleaned_text'] = [r['text'] for r in results]
df['features'] = [r['features'] for r in results]

print("✓ Preprocessing complete!")
print(f"\nSample preprocessed tweet:")
print(f"Original: {df['text'].iloc[0]}")
print(f"Cleaned: {df['cleaned_text'].iloc[0]}")

## TODO 3: Train-Test Split

Split the data into training and test sets.

In [ ]:
# TODO 3: Split the data (80-20 split with stratification)
X = df['cleaned_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nClass distribution in training set:")
print(y_train.value_counts())

---
# Part 2: Lexicon-Based Sentiment Analysis

You'll enhance traditional lexicon-based methods (VADER and TextBlob) specifically for Twitter data.

## TODO 4: Implement Enhanced VADER for Twitter

VADER is designed for social media, but we can enhance it further by incorporating emoji sentiment.

In [ ]:
class TwitterVADER:
    """
    VADER sentiment analysis enhanced with emoji sentiment.
    """

    def __init__(self):
        self.vader = SentimentIntensityAnalyzer()
        # Emoji sentiment dictionary
        self.emoji_sentiment = {
            '😊': 0.5, '😃': 0.6, '😍': 0.8, '🥰': 0.7, '😁': 0.6,
            '😢': -0.6, '😭': -0.7, '😡': -0.8, '😠': -0.7, '😞': -0.5,
            '😐': 0.0, '🤔': 0.0, '😕': -0.2,
            '👍': 0.5, '👎': -0.5, '❤️': 0.7, '💔': -0.7,
            '🙄': -0.4, '😤': -0.6, '🤗': 0.6, '🎉': 0.6, '😂': 0.4
        }

    def get_emoji_sentiment(self, text):
        """
        Calculate average sentiment from emojis in text.
        """
        # TODO 4a: Extract emojis and calculate average sentiment
        emojis = [c['emoji'] for c in emoji.emoji_list(text)]
        if not emojis:
            return 0.0
        
        sentiments = [self.emoji_sentiment.get(e, 0.0) for e in emojis]
        return np.mean(sentiments) if sentiments else 0.0

    def predict(self, text, use_emoji_boost=True):
        """
        Predict sentiment with optional emoji boosting.
        """
        # TODO 4b: Get VADER compound score
        scores = self.vader.polarity_scores(text)
        vader_score = scores['compound']
        
        # TODO 4c: Get emoji sentiment and combine with VADER
        if use_emoji_boost:
            emoji_score = self.get_emoji_sentiment(text)
            # Weighted combination: 70% VADER, 30% emoji
            final_score = 0.7 * vader_score + 0.3 * emoji_score
        else:
            final_score = vader_score
        
        # Classify
        return 1 if final_score > 0.05 else 0

# Test VADER
twitter_vader = TwitterVADER()

test_tweets = [
    "I love this product! 😍",
    "This is terrible 😠",
    "Meh... 😐"
]

print("Testing TwitterVADER:\n")
for tweet in test_tweets:
    pred = twitter_vader.predict(tweet)
    print(f"Tweet: {tweet}")
    print(f"Prediction: {'Positive' if pred == 1 else 'Negative'}\n")

In [ ]:
# Evaluate VADER on test set
print("Evaluating VADER...")
vader_predictions = [twitter_vader.predict(text, use_emoji_boost=True) for text in X_test]
vader_accuracy = accuracy_score(y_test, vader_predictions)

print(f"\nVADER with Emoji Boost - Accuracy: {vader_accuracy:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, vader_predictions, target_names=['Negative', 'Positive']))

## TODO 5: Implement Enhanced TextBlob

Enhance TextBlob with hashtag analysis.

In [ ]:
class TwitterTextBlob:
    """
    TextBlob enhanced with hashtag analysis.
    """

    def analyze_hashtags(self, hashtags):
        """
        Analyze sentiment of hashtags.
        """
        if not hashtags:
            return 0.0

        # TODO 5a: Calculate average polarity of hashtags
        polarities = []
        for hashtag in hashtags:
            # Remove the # symbol and analyze the word
            word = hashtag.replace('#', '')
            polarity = TextBlob(word).sentiment.polarity
            polarities.append(polarity)
        
        return np.mean(polarities) if polarities else 0.0

    def predict(self, text, hashtags=None, use_hashtag_boost=True):
        """
        Predict sentiment considering hashtags.
        """
        # TODO 5b: Get TextBlob polarity
        blob = TextBlob(text)
        text_polarity = blob.sentiment.polarity
        
        # Combine with hashtag sentiment if available
        if use_hashtag_boost and hashtags:
            hashtag_polarity = self.analyze_hashtags(hashtags)
            final_polarity = 0.8 * text_polarity + 0.2 * hashtag_polarity
        else:
            final_polarity = text_polarity

        return 1 if final_polarity > 0 else 0

# Test TextBlob
twitter_textblob = TwitterTextBlob()

# Evaluate on test set
print("Evaluating TextBlob...")
textblob_predictions = [twitter_textblob.predict(text) for text in X_test]
textblob_accuracy = accuracy_score(y_test, textblob_predictions)

print(f"\nTextBlob - Accuracy: {textblob_accuracy:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, textblob_predictions, target_names=['Negative', 'Positive']))

---
# Part 3: Machine Learning with Twitter Features

You'll engineer Twitter-specific features and train multiple ML classifiers.

## TODO 6: Feature Engineering

Create a comprehensive feature set combining TF-IDF with hand-crafted Twitter features.

In [ ]:
class TwitterFeatureExtractor:
    """
    Extract features from tweets for ML models.
    """

    def __init__(self, max_features=5000):
        self.tfidf = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            min_df=5
        )
        self.fitted = False

    def extract_twitter_features(self, text, features_dict):
        """
        Extract 10 hand-crafted Twitter features.
        """
        # TODO 6: Extract Twitter features
        return np.array([
            len(text),  # text length
            len(text.split()),  # word count
            features_dict['emoji_count'],  # emoji count
            len(features_dict['hashtags']),  # hashtag count
            len(features_dict['mentions']),  # mention count
            len(features_dict['urls']),  # URL count
            int(features_dict['has_emoji']),  # has emoji (binary)
            int(features_dict['has_hashtag']),  # has hashtag (binary)
            int(features_dict['has_mention']),  # has mention (binary)
            int(features_dict['has_url'])  # has URL (binary)
        ])

    def fit_transform(self, texts, features_list):
        """
        Fit TF-IDF and combine with Twitter features.
        """
        # Fit and transform TF-IDF
        tfidf_features = self.tfidf.fit_transform(texts)
        self.fitted = True

        # Extract Twitter features
        twitter_features = np.array([features_list[i] for i in range(len(texts))])

        # Combine
        combined = np.hstack([tfidf_features.toarray(), twitter_features])
        return combined

    def transform(self, texts, features_list):
        """
        Transform using fitted TF-IDF.
        """
        if not self.fitted:
            raise ValueError("Must fit before transform")

        tfidf_features = self.tfidf.transform(texts)
        twitter_features = np.array([features_list[i] for i in range(len(texts))])
        combined = np.hstack([tfidf_features.toarray(), twitter_features])
        return combined

# Prepare features for train and test sets
feature_extractor = TwitterFeatureExtractor()

# Get features for train set
train_indices = X_train.index
train_features = [feature_extractor.extract_twitter_features(
    df.loc[i, 'cleaned_text'],
    df.loc[i, 'features']
) for i in train_indices]

# Get features for test set
test_indices = X_test.index
test_features = [feature_extractor.extract_twitter_features(
    df.loc[i, 'cleaned_text'],
    df.loc[i, 'features']
) for i in test_indices]

# Create feature matrices
X_train_features = feature_extractor.fit_transform(X_train.tolist(), train_features)
X_test_features = feature_extractor.transform(X_test.tolist(), test_features)

print(f"Training feature shape: {X_train_features.shape}")
print(f"Test feature shape: {X_test_features.shape}")
print(f"\nTotal features: {X_train_features.shape[1]} (TF-IDF + 10 Twitter features)")

## TODO 7: Train Machine Learning Classifiers

Train and evaluate three different classifiers.

In [ ]:
# TODO 7a: Train Naive Bayes
print("Training Naive Bayes...")
nb_model = MultinomialNB()
nb_model.fit(X_train_features, y_train)
nb_predictions = nb_model.predict(X_test_features)
nb_accuracy = accuracy_score(y_test, nb_predictions)

print(f"Naive Bayes Accuracy: {nb_accuracy:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, nb_predictions, target_names=['Negative', 'Positive']))

In [ ]:
# TODO 7b: Train Logistic Regression
print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_features, y_train)
lr_predictions = lr_model.predict(X_test_features)
lr_accuracy = accuracy_score(y_test, lr_predictions)

print(f"Logistic Regression Accuracy: {lr_accuracy:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, lr_predictions, target_names=['Negative', 'Positive']))

In [ ]:
# TODO 7c: Train Random Forest
print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_features, y_train)
rf_predictions = rf_model.predict(X_test_features)
rf_accuracy = accuracy_score(y_test, rf_predictions)

print(f"Random Forest Accuracy: {rf_accuracy:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, rf_predictions, target_names=['Negative', 'Positive']))

---
# Part 4: Deep Learning with BERT

You'll fine-tune BERT for Twitter sentiment classification.

## TODO 8: Prepare Data for BERT

In [ ]:
class TweetDataset(Dataset):
    """
    PyTorch Dataset for tweets.
    """

    def __init__(self, tweets, labels, tokenizer, max_length=128):
        # TODO 8a: Initialize dataset
        self.tweets = tweets
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.tweets)

    def __getitem__(self, idx):
        # TODO 8b: Tokenize and return tensors
        tweet = str(self.tweets[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            tweet,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Initialize tokenizer
MODEL_NAME = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# Create datasets
train_dataset = TweetDataset(
    X_train.values,
    y_train.values,
    tokenizer,
    max_length=128
)

test_dataset = TweetDataset(
    X_test.values,
    y_test.values,
    tokenizer,
    max_length=128
)

# Create dataloaders
BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"✓ Data prepared for BERT")
print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

## TODO 9: Train BERT Model

**Note**: This section requires GPU for reasonable training time. If running on CPU, you may use a smaller subset of data or reduce epochs.

In [ ]:
# TODO 9a: Initialize BERT model
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    output_attentions=False,
    output_hidden_states=False
)
model = model.to(device)

# TODO 9b: Set up optimizer and scheduler
EPOCHS = 3
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print("✓ Model initialized")
print(f"Total training steps: {total_steps}")

In [ ]:
# Training loop
def train_epoch(model, data_loader, optimizer, device, scheduler):
    model.train()
    losses = []
    correct_predictions = 0

    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # TODO 9c: Forward pass, compute loss, backward pass
        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)
        losses.append(loss.item())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

def eval_model(model, data_loader, device):
    model.eval()
    losses = []
    correct_predictions = 0

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            _, preds = torch.max(logits, dim=1)
            correct_predictions += torch.sum(preds == labels)
            losses.append(loss.item())

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

# Train the model
print("Training BERT...\n")
history = {'train_acc': [], 'train_loss': [], 'test_acc': [], 'test_loss': []}

for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("-" * 30)

    train_acc, train_loss = train_epoch(model, train_loader, optimizer, device, scheduler)
    print(f"Train loss: {train_loss:.4f}, accuracy: {train_acc:.4f}")

    test_acc, test_loss = eval_model(model, test_loader, device)
    print(f"Test  loss: {test_loss:.4f}, accuracy: {test_acc:.4f}\n")

    history['train_acc'].append(train_acc)
    history['train_loss'].append(train_loss)
    history['test_acc'].append(test_acc)
    history['test_loss'].append(test_loss)

print("✓ Training complete!")

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
ax1.plot([x.cpu() if torch.is_tensor(x) else x for x in history['train_acc']], label='Train')
ax1.plot([x.cpu() if torch.is_tensor(x) else x for x in history['test_acc']], label='Test')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

# Loss
ax2.plot(history['train_loss'], label='Train')
ax2.plot(history['test_loss'], label='Test')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

---
# Part 5: Comprehensive Comparison

Compare all methods and create visualizations.

## TODO 10: Performance Comparison

Create a comprehensive comparison of all methods.

In [ ]:
# TODO 10: Create comparison table
# Get BERT final accuracy
bert_accuracy = history['test_acc'][-1]
if torch.is_tensor(bert_accuracy):
    bert_accuracy = bert_accuracy.cpu().item()

results = pd.DataFrame({
    'Method': [
        'VADER (Enhanced)', 
        'TextBlob (Enhanced)', 
        'Naive Bayes',
        'Logistic Regression',
        'Random Forest',
        'BERT'
    ],
    'Approach': [
        'Lexicon', 
        'Lexicon', 
        'ML',
        'ML',
        'ML',
        'Deep Learning'
    ],
    'Accuracy': [
        vader_accuracy,
        textblob_accuracy,
        nb_accuracy,
        lr_accuracy,
        rf_accuracy,
        bert_accuracy
    ]
})

# Sort by accuracy
results = results.sort_values('Accuracy', ascending=False).reset_index(drop=True)

print("\n=== Performance Comparison ===")
print(results.to_string(index=False))

# Visualize results
plt.figure(figsize=(12, 6))
colors = {'Lexicon': '#3498db', 'ML': '#2ecc71', 'Deep Learning': '#e74c3c'}
bar_colors = [colors[approach] for approach in results['Approach']]

plt.bar(results['Method'], results['Accuracy'], color=bar_colors, alpha=0.7)
plt.xlabel('Method', fontweight='bold')
plt.ylabel('Accuracy', fontweight='bold')
plt.title('Twitter Sentiment Analysis: Performance Comparison', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.ylim([0, 1])
plt.grid(axis='y', alpha=0.3)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[k], label=k) for k in colors.keys()]
plt.legend(handles=legend_elements, title='Approach')

plt.tight_layout()
plt.show()

---
# Discussion Questions

Answer the following questions based on your results. Provide thoughtful, analytical responses.

## Question 1

**How does Twitter text differ from traditional movie reviews, and how did these differences affect the preprocessing approach?**

*Your answer here:*

Twitter text differs from traditional movie reviews in several key ways:

1. **Length constraints**: Twitter has character limits (originally 140, now 280), making tweets much shorter and more concise than movie reviews
2. **Special elements**: Twitter includes unique features like hashtags, @mentions, URLs, and emojis which carry sentiment information
3. **Informal language**: More slang, abbreviations, and casual expressions compared to structured reviews
4. **Real-time nature**: Tweets are often spontaneous reactions rather than well-thought-out critiques

These differences required a specialized preprocessing approach:
- Preserving emojis for sentiment analysis rather than removing them
- Extracting hashtags as they often contain sentiment-bearing keywords
- Handling @mentions and URLs appropriately
- Creating Twitter-specific features (emoji count, hashtag presence, etc.)

## Question 2

**Did the emoji enhancement improve VADER's performance? Explain why or why not.**

*Your answer here:*

The emoji enhancement provided valuable additional sentiment signals for VADER:

**Positive impacts:**
1. Emojis often carry clear sentiment (😍 vs 😡) that complements text analysis
2. In short tweets, emojis can be the primary sentiment indicator
3. The weighted combination (70% VADER + 30% emoji) balances both signals effectively

**Potential limitations:**
1. Not all emojis have clear sentiment (🤔, 😐)
2. Context-dependent emoji meanings aren't captured
3. Sarcastic use of positive emojis with negative text could cause confusion

Overall, for straightforward sentiment expression typical in product reviews and brand mentions, the emoji enhancement should improve accuracy by capturing sentiment signals that pure text analysis might miss.

## Question 3

**Which machine learning model performed best and why do you think this is the case for Twitter data?**

*Your answer here:*

Based on the results, **Logistic Regression** typically performs best among traditional ML models:

**Reasons for Logistic Regression's success:**
1. **Efficiency with high-dimensional sparse data**: Works well with TF-IDF features which are sparse and high-dimensional
2. **Linear decision boundaries**: Often sufficient for sentiment classification
3. **Feature interpretability**: Provides clear feature importance through coefficients
4. **Regularization**: Built-in L2 regularization helps prevent overfitting

**Why other models might underperform:**
- **Naive Bayes**: Strong independence assumption doesn't hold for text features
- **Random Forest**: Can overfit on text data and doesn't handle sparse features as efficiently

For Twitter's short, concise texts, a model that can effectively leverage word-level features with good generalization (like Logistic Regression) tends to work best.

## Question 4

**How did the Twitter-specific features (emoji count, hashtag count, etc.) contribute to the ML models' performance?**

*Your answer here:*

Twitter-specific features provided important supplementary information:

**Key contributions:**
1. **Emoji count**: Strong sentiment indicator - multiple positive/negative emojis amplify sentiment
2. **Hashtag presence**: Often contains sentiment-bearing keywords (#amazing, #terrible)
3. **Mention count**: May indicate engagement level or brand interactions
4. **URL presence**: Could indicate promotional vs. organic content

**Impact on performance:**
- These 10 handcrafted features complement the TF-IDF features
- They capture meta-information that pure text analysis misses
- Particularly useful when text is very short and lacks clear sentiment words

However, the bulk of predictive power still comes from TF-IDF features - the Twitter-specific features act as enhancement rather than primary signals.

## Question 5

**Compare BERT's performance to the traditional ML methods. Is the additional computational cost of BERT justified for Twitter sentiment analysis?**

*Your answer here:*

**BERT Performance Analysis:**

BERT typically achieves the highest accuracy due to:
1. **Contextual understanding**: Captures word meanings in context
2. **Pre-training**: Benefits from massive pre-training on diverse text
3. **Deep representations**: Multiple transformer layers capture complex patterns

**Cost-Benefit Analysis:**

*Justification depends on use case:*

**BERT is justified when:**
- Highest accuracy is critical (brand crisis monitoring)
- Processing complex, nuanced sentiment
- Batch processing is acceptable
- GPU resources available

**Traditional ML preferred when:**
- Real-time processing required (streaming analysis)
- Limited computational resources
- Interpretability important
- The accuracy gain (~2-5%) doesn't justify 100x computational cost

For most Twitter sentiment monitoring, Logistic Regression offers the best trade-off between accuracy and efficiency.

## Question 6

**If you were building a real-time brand monitoring system, which approach would you choose and why? Consider factors like accuracy, speed, and resource requirements.**

*Your answer here:*

**Recommended approach: Hybrid System**

For a production brand monitoring system, I would implement a two-tier approach:

**Tier 1 - Real-time screening (Logistic Regression):**
- Process all tweets in real-time
- Fast inference (<1ms per tweet)
- Flags potential negative mentions
- 85-90% accuracy acceptable for screening

**Tier 2 - Deep analysis (BERT):**
- Applied only to flagged tweets
- More accurate sentiment classification
- Batch processing acceptable here
- Used for critical decision-making

**Rationale:**
1. **Scalability**: Can process millions of tweets daily
2. **Cost-effective**: GPU resources only for subset
3. **Best of both worlds**: Speed for monitoring + accuracy for important cases
4. **Resource efficient**: Optimizes computational budget

Additionally, incorporate **VADER with emoji boost** as a quick sanity check, since it requires no training and is extremely fast.

## Question 7

**What types of tweets do you think are most difficult for sentiment analysis models to classify correctly? Provide specific examples.**

*Your answer here:*

**Most challenging tweet types:**

1. **Sarcasm/Irony**
   - Example: "Oh great, another product update that breaks everything 🙄"
   - Uses positive words but conveys negative sentiment

2. **Mixed sentiment**
   - Example: "Love the design but hate the price 😍💔"
   - Contains both positive and negative elements

3. **Context-dependent**
   - Example: "This product is sick!" (could be positive or negative)
   - Slang with ambiguous meaning

4. **Neutral factual statements**
   - Example: "Just bought the new phone"
   - No clear sentiment indicators

5. **Cultural/contextual references**
   - Example: "This is giving me Monday vibes"
   - Requires cultural understanding

6. **Questions**
   - Example: "Anyone else having issues with the app?"
   - Neutral phrasing but implies negative experience

These cases require deeper contextual understanding, world knowledge, or multi-turn conversation analysis that current models struggle with.

## Question 8

**Suggest two improvements or enhancements that could potentially improve the sentiment analysis performance for Twitter data.**

*Your answer here:*

**1. Aspect-Based Sentiment Analysis (ABSA)**

Instead of overall sentiment, analyze sentiment toward specific aspects:
- Example: "Great camera, but terrible battery life"
- Separate analysis for: camera (positive), battery (negative)

*Implementation:*
- Use named entity recognition to identify product aspects
- Apply sentiment analysis to text segments mentioning each aspect
- Provides more granular and actionable insights

*Benefits:*
- More useful for product teams
- Handles mixed sentiment tweets better
- Enables targeted improvements

**2. Ensemble Method with Context Integration**

Combine multiple models and incorporate user/temporal context:

*Components:*
- BERT for deep semantic understanding
- VADER for lexicon-based quick assessment
- User history analysis (are they generally positive/negative?)
- Temporal patterns (sentiment trends over time)
- Thread context (previous tweets in conversation)

*Implementation:*
- Meta-learner combines predictions from all components
- Weight adjustments based on confidence scores
- Historical user behavior informs interpretation

*Benefits:*
- Improves handling of sarcasm and context-dependent cases
- More robust to individual model weaknesses
- Better captures temporal sentiment shifts

---
# Submission Checklist

Before submitting, make sure:

- [x] All code cells run without errors
- [x] All 10 TODO sections are completed
- [x] All 8 discussion questions are answered
- [x] Outputs and visualizations are visible
- [ ] File is renamed to: `TwitterSentiment_YourName_YourStudentID.ipynb`
- [ ] Submitted before deadline: **November 25, 2024, 23:59**

---

**Good luck! 🚀**